# 04 — Backpropagation: Full Manual Derivation

**Goal:** Walk through backprop for a 3-layer network step-by-step, no autograd. Derive every gradient by hand, implement it, and verify with numerical gradient checking.

---

## Network Architecture

3-layer network for multiclass classification:

$$x \xrightarrow{W_1, b_1} z_1 \xrightarrow{\text{ReLU}} a_1 \xrightarrow{W_2, b_2} z_2 \xrightarrow{\text{ReLU}} a_2 \xrightarrow{W_3, b_3} z_3 \xrightarrow{\text{softmax}} \hat{y} \xrightarrow{\text{CE}} L$$

**Dimensions:** Input $x \in \mathbb{R}^{N \times D}$, hidden layers of width $H$, output $C$ classes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
%matplotlib inline

## 1. Forward Pass — Step by Step

For a single training batch $(X, y)$:

| Step | Operation | Shape |
|------|-----------|-------|
| 1 | $z_1 = X W_1 + b_1$ | $(N, H_1)$ |
| 2 | $a_1 = \text{ReLU}(z_1)$ | $(N, H_1)$ |
| 3 | $z_2 = a_1 W_2 + b_2$ | $(N, H_2)$ |
| 4 | $a_2 = \text{ReLU}(z_2)$ | $(N, H_2)$ |
| 5 | $z_3 = a_2 W_3 + b_3$ | $(N, C)$ |
| 6 | $\hat{y} = \text{softmax}(z_3)$ | $(N, C)$ |
| 7 | $L = -\frac{1}{N} \sum_{i} \log \hat{y}_{i, y_i}$ | scalar |

In [ ]:
# Network dimensions
D = 4    # input features
H1 = 8   # hidden layer 1
H2 = 6   # hidden layer 2
C = 3    # output classes
N = 5    # batch size

# Initialize parameters (He init for ReLU)
np.random.seed(42)
W1 = np.random.randn(D, H1) * np.sqrt(2.0 / D)
b1 = np.zeros((1, H1))
W2 = np.random.randn(H1, H2) * np.sqrt(2.0 / H1)
b2 = np.zeros((1, H2))
W3 = np.random.randn(H2, C) * np.sqrt(2.0 / H2)
b3 = np.zeros((1, C))

# Synthetic data
X = np.random.randn(N, D)
y = np.array([0, 1, 2, 1, 0])  # class labels

# ====== FORWARD PASS ======
# Layer 1
z1 = X @ W1 + b1          # (N, H1)
a1 = np.maximum(0, z1)     # ReLU

# Layer 2
z2 = a1 @ W2 + b2          # (N, H2)
a2 = np.maximum(0, z2)     # ReLU

# Layer 3 (output)
z3 = a2 @ W3 + b3          # (N, C)

# Softmax
z3_shifted = z3 - np.max(z3, axis=1, keepdims=True)  # numerical stability
exp_scores = np.exp(z3_shifted)
probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)  # (N, C)

# Cross-entropy loss
correct_log_probs = -np.log(probs[np.arange(N), y] + 1e-12)
loss = np.mean(correct_log_probs)

print("Forward pass complete.")
print(f"  z1 shape: {z1.shape}")
print(f"  a1 shape: {a1.shape}")
print(f"  z2 shape: {z2.shape}")
print(f"  a2 shape: {a2.shape}")
print(f"  z3 shape: {z3.shape}")
print(f"  probs shape: {probs.shape}")
print(f"  Loss: {loss:.4f}")

## 2. Backward Pass — Deriving Every Gradient

### Step 7 -> 5: Gradient of Softmax + Cross-Entropy

The combined gradient of softmax + cross-entropy simplifies to:

$$\frac{\partial L}{\partial z_3} = \frac{1}{N} (\hat{y} - \mathbf{1}_{\text{hot}}(y))$$

where $\mathbf{1}_{\text{hot}}(y)$ is the one-hot encoded target.

**Derivation sketch:**
- $L = -\frac{1}{N}\sum_i \log(\hat{y}_{i,y_i})$
- $\hat{y}_{i,j} = \frac{e^{z_{3_{i,j}}}}{\sum_k e^{z_{3_{i,k}}}}$
- For $j = y_i$: $\frac{\partial L}{\partial z_{3_{i,j}}} = \frac{1}{N}(\hat{y}_{i,j} - 1)$
- For $j \neq y_i$: $\frac{\partial L}{\partial z_{3_{i,j}}} = \frac{1}{N}\hat{y}_{i,j}$

In [ ]:
# ====== BACKWARD PASS ======

# Step 1: dL/dz3 (softmax + CE gradient)
dz3 = probs.copy()                         # (N, C)
dz3[np.arange(N), y] -= 1.0                # subtract 1 from correct class
dz3 /= N                                    # average over batch

print(f"dz3 shape: {dz3.shape}")
print(f"dz3 (first sample): {dz3[0]}")
print(f"Sum of dz3 per sample (should be ~0): {np.sum(dz3, axis=1)}")

### Step 5: Gradients for Layer 3 ($z_3 = a_2 W_3 + b_3$)

Given $\frac{\partial L}{\partial z_3}$, compute:

$$\frac{\partial L}{\partial W_3} = a_2^T \cdot \frac{\partial L}{\partial z_3} \quad (H_2 \times C)$$

$$\frac{\partial L}{\partial b_3} = \mathbf{1}^T \cdot \frac{\partial L}{\partial z_3} \quad (1 \times C)$$

$$\frac{\partial L}{\partial a_2} = \frac{\partial L}{\partial z_3} \cdot W_3^T \quad (N \times H_2)$$

In [ ]:
# Step 2: Gradients for layer 3 parameters
dW3 = a2.T @ dz3           # (H2, N) @ (N, C) = (H2, C)
db3 = np.sum(dz3, axis=0, keepdims=True)  # (1, C)
da2 = dz3 @ W3.T           # (N, C) @ (C, H2) = (N, H2)

print(f"dW3 shape: {dW3.shape}")
print(f"db3 shape: {db3.shape}")
print(f"da2 shape: {da2.shape}")

### Step 4: Backprop through ReLU ($a_2 = \text{ReLU}(z_2)$)

$$\frac{\partial L}{\partial z_2} = \frac{\partial L}{\partial a_2} \odot \mathbf{1}_{z_2 > 0}$$

ReLU passes the gradient through where $z_2 > 0$ and blocks it where $z_2 \leq 0$.

In [ ]:
# Step 3: Backprop through ReLU
dz2 = da2 * (z2 > 0).astype(float)   # (N, H2)

print(f"dz2 shape: {dz2.shape}")
print(f"Fraction of z2 > 0: {np.mean(z2 > 0):.2f}")
print(f"(This fraction of gradients passes through)")

### Steps 3-2: Layer 2 gradients and ReLU

In [ ]:
# Step 4: Gradients for layer 2 parameters
dW2 = a1.T @ dz2           # (H1, N) @ (N, H2) = (H1, H2)
db2 = np.sum(dz2, axis=0, keepdims=True)  # (1, H2)
da1 = dz2 @ W2.T           # (N, H2) @ (H2, H1) = (N, H1)

# Step 5: Backprop through ReLU for layer 1
dz1 = da1 * (z1 > 0).astype(float)   # (N, H1)

# Step 6: Gradients for layer 1 parameters
dW1 = X.T @ dz1            # (D, N) @ (N, H1) = (D, H1)
db1 = np.sum(dz1, axis=0, keepdims=True)  # (1, H1)
dX = dz1 @ W1.T            # (N, H1) @ (H1, D) = (N, D) -- gradient w.r.t. input

print("All gradients computed:")
for name, grad in [('dW1', dW1), ('db1', db1), ('dW2', dW2), ('db2', db2), 
                    ('dW3', dW3), ('db3', db3)]:
    print(f"  {name}: shape={grad.shape}, norm={np.linalg.norm(grad):.6f}")

## 3. Numerical Gradient Checking

The gold standard for verifying backprop: compute $\frac{\partial L}{\partial \theta_i} \approx \frac{f(\theta_i + \epsilon) - f(\theta_i - \epsilon)}{2\epsilon}$

**Rule of thumb:** relative error should be < $10^{-5}$ for double precision.

In [ ]:
def forward_pass(X, y, W1, b1, W2, b2, W3, b3):
    """Full forward pass, returns loss."""
    N = X.shape[0]
    z1 = X @ W1 + b1
    a1 = np.maximum(0, z1)
    z2 = a1 @ W2 + b2
    a2 = np.maximum(0, z2)
    z3 = a2 @ W3 + b3
    z3_shifted = z3 - np.max(z3, axis=1, keepdims=True)
    exp_scores = np.exp(z3_shifted)
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    loss = np.mean(-np.log(probs[np.arange(N), y] + 1e-12))
    return loss


def numerical_gradient(f, param, eps=1e-5):
    """Compute numerical gradient for every element of param."""
    grad = np.zeros_like(param)
    it = np.nditer(param, flags=['multi_index'], op_flags=['readwrite'])
    while not it.finished:
        idx = it.multi_index
        old_val = param[idx]
        
        param[idx] = old_val + eps
        loss_plus = f()
        
        param[idx] = old_val - eps
        loss_minus = f()
        
        grad[idx] = (loss_plus - loss_minus) / (2 * eps)
        param[idx] = old_val
        it.iternext()
    return grad


# Check each parameter
params_and_grads = [
    ('W1', W1, dW1), ('b1', b1, db1),
    ('W2', W2, dW2), ('b2', b2, db2),
    ('W3', W3, dW3), ('b3', b3, db3),
]

print(f"{'Param':<6} {'Max Rel Error':>15} {'Status':>8}")
print("-" * 35)

for name, param, analytical_grad in params_and_grads:
    f = lambda: forward_pass(X, y, W1, b1, W2, b2, W3, b3)
    num_grad = numerical_gradient(f, param)
    
    # Relative error
    rel_error = np.max(np.abs(analytical_grad - num_grad) / 
                       (np.maximum(np.abs(analytical_grad) + np.abs(num_grad), 1e-8)))
    status = 'OK' if rel_error < 1e-5 else 'FAIL'
    print(f"{name:<6} {rel_error:>15.2e} {status:>8}")

## 4. Visualize Gradient Flow

In [ ]:
# Visualize gradient norms at each layer
gradient_norms = {
    'dz3 (output)': np.linalg.norm(dz3),
    'dW3': np.linalg.norm(dW3),
    'da2': np.linalg.norm(da2),
    'dz2 (after ReLU)': np.linalg.norm(dz2),
    'dW2': np.linalg.norm(dW2),
    'da1': np.linalg.norm(da1),
    'dz1 (after ReLU)': np.linalg.norm(dz1),
    'dW1': np.linalg.norm(dW1),
}

fig, ax = plt.subplots(figsize=(10, 4))
names = list(gradient_norms.keys())
values = list(gradient_norms.values())
colors = ['#FF6B6B' if 'dW' in n else '#4ECDC4' for n in names]

bars = ax.barh(range(len(names)), values, color=colors, edgecolor='black', alpha=0.7)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.set_xlabel('Gradient Norm')
ax.set_title('Gradient Norms Through the Network (red = weight grads, teal = activation grads)')
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Complete Backprop — Wrapped as a Function

In [ ]:
def forward_backward(X, y, W1, b1, W2, b2, W3, b3):
    """Complete forward + backward pass. Returns loss, gradients, and caches."""
    N = X.shape[0]
    
    # ---- Forward ----
    z1 = X @ W1 + b1
    a1 = np.maximum(0, z1)
    z2 = a1 @ W2 + b2
    a2 = np.maximum(0, z2)
    z3 = a2 @ W3 + b3
    
    # Softmax
    z3_shifted = z3 - np.max(z3, axis=1, keepdims=True)
    exp_scores = np.exp(z3_shifted)
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    loss = np.mean(-np.log(probs[np.arange(N), y] + 1e-12))
    
    # ---- Backward ----
    dz3 = probs.copy()
    dz3[np.arange(N), y] -= 1.0
    dz3 /= N
    
    dW3 = a2.T @ dz3
    db3 = np.sum(dz3, axis=0, keepdims=True)
    da2 = dz3 @ W3.T
    
    dz2 = da2 * (z2 > 0).astype(float)
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)
    da1 = dz2 @ W2.T
    
    dz1 = da1 * (z1 > 0).astype(float)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)
    
    grads = {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2, 'W3': dW3, 'b3': db3}
    return loss, grads, probs

print("forward_backward function defined.")

## 6. Train the Network

In [ ]:
# Generate a proper dataset
np.random.seed(42)
N_train = 300
D_in = 2
C_out = 3

# Spiral data
X_train = np.zeros((N_train * C_out, D_in))
y_train = np.zeros(N_train * C_out, dtype=int)
for c in range(C_out):
    idx = range(N_train * c, N_train * (c + 1))
    r = np.linspace(0.0, 1.0, N_train)
    t = np.linspace(c * 4, (c + 1) * 4, N_train) + np.random.randn(N_train) * 0.2
    X_train[idx] = np.c_[r * np.sin(t), r * np.cos(t)]
    y_train[idx] = c

# Initialize
np.random.seed(42)
H = 64
W1 = np.random.randn(D_in, H) * np.sqrt(2.0 / D_in)
b1 = np.zeros((1, H))
W2 = np.random.randn(H, H) * np.sqrt(2.0 / H)
b2 = np.zeros((1, H))
W3 = np.random.randn(H, C_out) * np.sqrt(2.0 / H)
b3 = np.zeros((1, C_out))

# Training loop
lr = 0.5
losses = []
accs = []

for epoch in range(300):
    loss, grads, probs = forward_backward(X_train, y_train, W1, b1, W2, b2, W3, b3)
    losses.append(loss)
    
    preds = np.argmax(probs, axis=1)
    acc = np.mean(preds == y_train)
    accs.append(acc)
    
    # SGD update
    W1 -= lr * grads['W1']
    b1 -= lr * grads['b1']
    W2 -= lr * grads['W2']
    b2 -= lr * grads['b2']
    W3 -= lr * grads['W3']
    b3 -= lr * grads['b3']
    
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d}: loss={loss:.4f}, acc={acc:.3f}")

print(f"\nFinal: loss={losses[-1]:.4f}, acc={accs[-1]:.3f}")

In [ ]:
# Plot training curves and decision boundary
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(losses)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss'); axes[0].grid(True, alpha=0.3)

axes[1].plot(accs)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy'); axes[1].grid(True, alpha=0.3)

# Decision boundary
h = 0.02
x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid = np.c_[xx.ravel(), yy.ravel()]

z1_g = grid @ W1 + b1
a1_g = np.maximum(0, z1_g)
z2_g = a1_g @ W2 + b2
a2_g = np.maximum(0, z2_g)
z3_g = a2_g @ W3 + b3
Z = np.argmax(z3_g, axis=1).reshape(xx.shape)

from matplotlib.colors import ListedColormap
axes[2].contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#FF6B6B', '#4ECDC4', '#45B7D1']))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
for c in range(3):
    mask = y_train == c
    axes[2].scatter(X_train[mask, 0], X_train[mask, 1], c=colors[c], s=10, edgecolors='k', linewidth=0.3)
axes[2].set_title('Decision Boundary'); axes[2].set_aspect('equal')

plt.tight_layout()
plt.show()

## 7. Vanishing/Exploding Gradients — Demonstration

In [ ]:
# Deep network gradient flow analysis
def analyze_gradient_flow(n_layers, activation='relu', init_std=None):
    """Build a deep net and track gradient norms per layer."""
    np.random.seed(42)
    width = 32
    N = 16
    x = np.random.randn(N, width)
    
    # Forward
    Ws, zs, acts = [], [], [x]
    h = x
    for i in range(n_layers):
        if init_std is None:
            std = np.sqrt(2.0 / width) if activation == 'relu' else np.sqrt(1.0 / width)
        else:
            std = init_std
        W = np.random.randn(width, width) * std
        Ws.append(W)
        z = h @ W
        zs.append(z)
        if activation == 'relu':
            h = np.maximum(0, z)
        elif activation == 'sigmoid':
            h = 1.0 / (1.0 + np.exp(-z))
        elif activation == 'tanh':
            h = np.tanh(z)
        acts.append(h)
    
    # Final linear + softmax
    W_out = np.random.randn(width, 3) * 0.01
    logits = h @ W_out
    targets = np.random.randint(0, 3, N)
    
    # Backward from softmax + CE
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    probs = np.exp(shifted) / np.sum(np.exp(shifted), axis=1, keepdims=True)
    dlogits = probs.copy()
    dlogits[np.arange(N), targets] -= 1.0
    dlogits /= N
    
    dh = dlogits @ W_out.T
    grad_norms = []
    
    for i in range(n_layers - 1, -1, -1):
        # Through activation
        if activation == 'relu':
            dz = dh * (zs[i] > 0).astype(float)
        elif activation == 'sigmoid':
            s = 1.0 / (1.0 + np.exp(-zs[i]))
            dz = dh * s * (1 - s)
        elif activation == 'tanh':
            t = np.tanh(zs[i])
            dz = dh * (1 - t**2)
        
        grad_norms.append(np.linalg.norm(dz))
        dh = dz @ Ws[i].T
    
    return grad_norms[::-1]  # from input to output


fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Activation comparison
for act in ['relu', 'tanh', 'sigmoid']:
    norms = analyze_gradient_flow(20, activation=act)
    axes[0].plot(norms, label=act, linewidth=2)
axes[0].set_yscale('log')
axes[0].set_xlabel('Layer'); axes[0].set_ylabel('Gradient Norm')
axes[0].set_title('Gradient Flow: Activation Comparison (20 layers)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# 2. Init scale matters (ReLU)
for std in [0.001, 0.01, None, 0.1, 1.0]:
    label = f'std={std}' if std else 'He init'
    norms = analyze_gradient_flow(15, activation='relu', init_std=std)
    axes[1].plot(norms, label=label, linewidth=2)
axes[1].set_yscale('log')
axes[1].set_xlabel('Layer'); axes[1].set_ylabel('Gradient Norm')
axes[1].set_title('Gradient Flow: Init Scale Matters (ReLU)')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# 3. Depth matters
for depth in [5, 10, 20, 50]:
    norms = analyze_gradient_flow(depth, activation='sigmoid')
    axes[2].plot(range(depth), norms, label=f'{depth} layers', linewidth=2)
axes[2].set_yscale('log')
axes[2].set_xlabel('Layer'); axes[2].set_ylabel('Gradient Norm')
axes[2].set_title('Sigmoid: Deeper = Worse Vanishing')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Derive: Softmax + Cross-Entropy Gradient

This is a classic interview derivation. Let's work through it carefully.

**Setup:** $p_j = \frac{e^{z_j}}{\sum_k e^{z_k}}$ (softmax), $L = -\log(p_y)$ (CE for correct class $y$).

**Need:** $\frac{\partial L}{\partial z_i}$ for all $i$.

**Case 1: $i = y$ (correct class)**
$$\frac{\partial L}{\partial z_y} = -\frac{1}{p_y} \cdot \frac{\partial p_y}{\partial z_y} = -\frac{1}{p_y} \cdot p_y(1 - p_y) = -(1 - p_y) = p_y - 1$$

**Case 2: $i \neq y$ (wrong class)**
$$\frac{\partial L}{\partial z_i} = -\frac{1}{p_y} \cdot \frac{\partial p_y}{\partial z_i} = -\frac{1}{p_y} \cdot (-p_y \cdot p_i) = p_i$$

**Combined:** $\frac{\partial L}{\partial z_i} = p_i - \mathbb{1}_{i = y}$

This is why the code is just `dz = probs.copy(); dz[target] -= 1`.

In [ ]:
# Verify the derivation numerically
np.random.seed(42)
z = np.random.randn(1, 5)  # 5 classes
y_label = 2  # correct class

# Analytical gradient
shifted = z - np.max(z)
p = np.exp(shifted) / np.sum(np.exp(shifted))
dz_analytical = p.copy()
dz_analytical[0, y_label] -= 1.0

# Numerical gradient
eps = 1e-5
dz_numerical = np.zeros_like(z)
for i in range(5):
    z_plus = z.copy(); z_plus[0, i] += eps
    z_minus = z.copy(); z_minus[0, i] -= eps
    
    p_plus = np.exp(z_plus - np.max(z_plus)) / np.sum(np.exp(z_plus - np.max(z_plus)))
    p_minus = np.exp(z_minus - np.max(z_minus)) / np.sum(np.exp(z_minus - np.max(z_minus)))
    
    loss_plus = -np.log(p_plus[0, y_label])
    loss_minus = -np.log(p_minus[0, y_label])
    dz_numerical[0, i] = (loss_plus - loss_minus) / (2 * eps)

print("Softmax + CE gradient verification:")
print(f"  Analytical: {dz_analytical[0]}")
print(f"  Numerical:  {dz_numerical[0]}")
print(f"  Match: {np.allclose(dz_analytical, dz_numerical, atol=1e-6)}")

## 9. Gradient Checking Tips for Interviews

| Tip | Why |
|-----|-----|
| Use **centered** differences: $(f(x+\epsilon) - f(x-\epsilon)) / 2\epsilon$ | $O(\epsilon^2)$ error vs $O(\epsilon)$ for one-sided |
| Use $\epsilon \approx 10^{-5}$ | Too small => numerical issues; too large => bad approximation |
| Check **relative** error, not absolute | Scale-invariant comparison |
| Turn off dropout and batch norm | These introduce randomness that breaks the check |
| Check with small networks first | Faster and easier to debug |

## Interview Quick Reference

**Q: Walk me through backprop for a 3-layer network.**  
1. Forward: compute $z_l = a_{l-1} W_l + b_l$, then $a_l = \sigma(z_l)$ for each layer
2. Compute loss (softmax + CE at output)
3. Backward from loss: $\frac{\partial L}{\partial z_3} = p - y_{\text{onehot}}$
4. For each layer going back: $\frac{\partial L}{\partial W_l} = a_{l-1}^T \cdot \frac{\partial L}{\partial z_l}$, $\frac{\partial L}{\partial a_{l-1}} = \frac{\partial L}{\partial z_l} \cdot W_l^T$
5. Through activation: element-wise multiply by activation derivative

**Q: Derive softmax + CE gradient.**  
$\frac{\partial L}{\partial z_i} = p_i - \mathbb{1}_{i=y}$ (see derivation above).

**Q: Vanishing gradients — causes and fixes?**  
Causes: sigmoid/tanh saturation (max grad < 1, multiplied through layers). Fixes: ReLU, skip connections (ResNet), normalization, better init (He/Xavier), gradient clipping.

**Q: Exploding gradients?**  
Gradient norms grow exponentially (eigenvalues of weight matrices > 1). Fixes: gradient clipping, weight initialization, normalization layers.

---
*Next: 05_normalization.ipynb — BatchNorm, LayerNorm, and friends*